# Clase 154 — Agentes (ReAct + multi-agent)

Implementamos un **loop ReAct** desde scratch (sin LLM real, usando un mock determinístico) y luego **2 agentes coordinados por un manager**.

In [ ]:
import re, random
random.seed(42)

## 1. Tools

In [ ]:
def calculator(expr):
    """Evalúa expresiones aritméticas simples (whitelist)."""
    if not re.fullmatch(r'[\d+\-*/().\s]+', expr):
        return 'ERROR: chars no permitidos'
    try: return str(eval(expr))
    except Exception as e: return f'ERROR: {e}'

MOCK_FACTS = {
    'populación de argentina': '46 millones (2023)',
    'populación de japón': '125 millones (2023)',
    'capital de francia': 'París',
    'altura del everest': '8849 metros',
}
def search(query):
    q = query.lower().strip('?.! ')
    for k, v in MOCK_FACTS.items():
        if k in q: return v
    return 'sin resultado'

TOOLS = {'calculator': calculator, 'search': search}
print(calculator('12*7'), '|', search('populación de Argentina'))

## 2. Mock LLM con reglas (genera Thought/Action/Observation)

En producción esto sería un LLM real al que se le pasa el historial.

In [ ]:
def mock_llm(question, trace):
    """Devuelve siguiente paso como dict {thought, action, action_input} o {answer}."""
    done = [step['action'] for step in trace if 'action' in step]
    # Heurística: si la pregunta tiene aritmética y aún no calculé → calc.
    has_math = bool(re.search(r'\d+\s*[+\-*/]\s*\d+', question))
    has_fact = any(k in question.lower() for k in MOCK_FACTS)
    if has_math and 'calculator' not in done:
        expr = re.search(r'[\d+\-*/().\s]+(?=\s*\+\s*populación|$|\?)', question)
        # Extrae primera expresión aritmética
        m = re.search(r'(\d+\s*[*/+\-]\s*\d+(?:\s*[*/+\-]\s*\d+)*)', question)
        return {'thought': 'Necesito calcular la parte aritmética primero.',
                'action': 'calculator', 'action_input': m.group(1)}
    if has_fact and 'search' not in done:
        for k in MOCK_FACTS:
            if k in question.lower():
                return {'thought': f'Necesito buscar: {k}.',
                        'action': 'search', 'action_input': k}
    # Si ya tengo todo → respuesta final componiendo observations.
    obs = [s['observation'] for s in trace if 'observation' in s]
    return {'thought': 'Ya tengo info suficiente.',
            'answer': ' + '.join(obs) if obs else 'no sé'}

## 3. Loop ReAct

In [ ]:
def react(question, max_steps=6):
    trace = []
    print(f'Q: {question}\n')
    for step in range(max_steps):
        out = mock_llm(question, trace)
        print(f'-- step {step} --')
        print(f'  Thought: {out["thought"]}')
        if 'answer' in out:
            print(f'  ✅ Answer: {out["answer"]}')
            return out['answer']
        action, inp = out['action'], out['action_input']
        print(f'  Action: {action}({inp!r})')
        obs = TOOLS[action](inp)
        print(f'  Observation: {obs}\n')
        trace.append({'action': action, 'input': inp, 'observation': obs})
    return 'max_steps exceeded'

react('¿Cuánto es 12*7 + populación de Argentina?')

## 4. Multi-agent: Researcher + Writer coordinados por Manager

In [ ]:
class Researcher:
    """Junta hechos relevantes via search."""
    def run(self, topic):
        facts = []
        for k in MOCK_FACTS:
            if any(w in k for w in topic.lower().split()):
                facts.append({'k': k, 'v': MOCK_FACTS[k]})
        return {'agent': 'researcher', 'facts': facts}

class Writer:
    """Convierte facts en párrafo."""
    def run(self, facts):
        if not facts: text = 'No se encontraron datos relevantes.'
        else:
            sents = [f'{f["k"].capitalize()}: {f["v"]}.' for f in facts]
            text = ' '.join(sents)
        return {'agent': 'writer', 'text': text}

class Manager:
    def __init__(self): self.researcher, self.writer = Researcher(), Writer()
    def run(self, topic):
        msgs = [{'from': 'user', 'content': topic}]
        r = self.researcher.run(topic); msgs.append({'from': 'researcher', 'content': r})
        w = self.writer.run(r['facts']); msgs.append({'from': 'writer', 'content': w})
        msgs.append({'from': 'manager', 'content': {'final': w['text']}})
        return msgs

msgs = Manager().run('populación argentina y japón')
for m in msgs: print(f'[{m["from"]:10s}] {m["content"]}')

## 5. Trace render

In [ ]:
def render_trace(msgs):
    print('=' * 60)
    for i, m in enumerate(msgs):
        prefix = '👤' if m['from'] == 'user' else '🤖'
        print(f'{prefix} [{i}] {m["from"]}')
        print(f'   → {m["content"]}')
    print('=' * 60)
render_trace(msgs)

## 6. Patrones avanzados

- **ReAct** (Yao 2022): Thought → Action → Observation, loop.
- **Reflexion**: el agente se auto-critica y reintenta.
- **Plan-and-Execute**: plan completo upfront, luego ejecuta.
- **Multi-agent**: researcher/writer/critic, supervisor/worker, debate.
- **Frameworks reales**: LangGraph, AutoGen, CrewAI, OpenAI Swarm.

**Riesgos**: loops infinitos, costo de tokens explosivo, tool hallucination.

## Ejercicio guiado

1. Agregar un **Critic agent** que rechaza la respuesta del writer si tiene <2 facts.
2. Implementar Reflexion: si la respuesta es 'sin resultado', re-prompt con hint.
3. Convertir tools en MCP (clase 153) y conectar.

## Conclusiones

- ReAct = pattern minimal y poderoso para tool use.
- Multi-agent: división de roles mejora calidad cuando la tarea es compleja.
- Siempre limitar `max_steps` y monitorear costo.